# Bayesian Belief - Fine-Tuning

Marginalizes posterior directly for role predictions (no value matrix needed).
- Each stage: role dist = posterior marginal for each player
- No stick-or-switch mechanism — pure posterior sampling every stage

Params: tau_prior, epsilon (2 params)

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## Model Definition

In [3]:
def posterior_marginal(prior, agent_i):
    """Marginalize the (3,3,3) joint posterior to get P(role) for agent_i."""
    marg = np.sum(prior, axis=tuple(j for j in range(3) if j != agent_i))
    total = marg.sum()
    return marg / total if total > 0 else np.ones(3) / 3.0


def make_bayesian_belief(tau_prior=2.0, epsilon=0.2):
    """Bayesian Belief: posterior marginals directly as role predictions."""
    def predict_fn(record):
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            # Marginalize posterior for each player
            per_agent = [posterior_marginal(prior, i) for i in range(3)]

            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            # Teacher-force: advance game with actual human roles
            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn


def precompute_trajectories(records, tau_prior, epsilon):
    """Precompute posterior + game state per stage for each record.

    Depends only on (tau_prior, epsilon) and observed human actions.
    Since Bayesian Belief has no switching params, the full prediction
    is determined by (tau_prior, epsilon) — but we keep this structure
    for consistency with switch-stage filtering.
    """
    trajectories = []
    for record in records:
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        turn_idx = 0
        stages = []

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            stages.append({
                'prior': prior.copy(),
                'human_combo': human_combo,
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent_t = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent_t, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent_t, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent_t, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        trajectories.append(stages)
    return trajectories


def predict_from_trajectory(trajectory):
    """Generate predictions from precomputed trajectory."""
    results = []
    for stage in trajectory:
        prior = stage['prior']
        human_combo = stage['human_combo']

        per_agent = [posterior_marginal(prior, i) for i in range(3)]

        predicted_dist = {}
        for r0 in range(3):
            for r1 in range(3):
                for r2 in range(3):
                    combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                    predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

        results.append({
            'predicted_dist': predicted_dist,
            'human_combo': human_combo,
            'model_marginal': np.mean(per_agent, axis=0),
        })
    return results

## Fine-Tuning Helpers

In [4]:
metric = 'combo_r'

def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])


def evaluate_belief(records, tau_prior, epsilon):
    """Run Bayesian Belief at given params (uncached, for scipy)."""
    results = run_predictions(records, make_bayesian_belief(
        tau_prior=tau_prior, epsilon=epsilon))
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'epsilon': epsilon,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_from_cache(records, trajectories):
    """Evaluate using precomputed trajectories (fast)."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(trajectories[idx])
    results = run_predictions(records, predict_fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_belief(records, tau_prior_vals, eps_vals):
    """Grid search over (tau_prior, epsilon)."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    total = len(tau_prior_vals) * len(eps_vals)
    count = 0

    for tp in tau_prior_vals:
        for eps in eps_vals:
            count += 1
            if count % 50 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            trajectories = precompute_trajectories(records, tp, eps)
            res = evaluate_from_cache(records, trajectories)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

## Aggregate Fine-Tuning

In [5]:
# Grid ranges
tau_prior_min, tau_prior_max, tau_prior_steps = 0.1, 20.0, 10
eps_min, eps_max, eps_steps = 0.0, 1.0, 21

tau_prior_vals = np.linspace(tau_prior_min, tau_prior_max, tau_prior_steps)
eps_vals = np.linspace(eps_min, eps_max, eps_steps)

total = len(tau_prior_vals) * len(eps_vals)
print(f'Coarse grid: {len(tau_prior_vals)} x {len(eps_vals)} = {total} points')
print(f'  tau_prior: linspace({tau_prior_min}, {tau_prior_max}, {tau_prior_steps})')
print(f'  epsilon:   linspace({eps_min}, {eps_max}, {eps_steps})')
print()

sweep_results = grid_search_belief(records, tau_prior_vals, eps_vals)
best = pick_best(sweep_results, metric)
print(f'\nCoarse best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Coarse grid: 10 x 21 = 210 points
  tau_prior: linspace(0.1, 20.0, 10)
  epsilon:   linspace(0.0, 1.0, 21)



/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


  [50/210] ...
  [100/210] ...
  [150/210] ...
  [200/210] ...
  [210/210] ...

Coarse best: tau_prior=2.3111 eps=0.550000
  combo_r=0.4829  marg_r=0.3953  mean_ll=-2.6332


In [6]:
# Refine around coarse best
tp_step = (tau_prior_max - tau_prior_min) / tau_prior_steps
eps_step_size = (eps_max - eps_min) / (eps_steps - 1)

fine_tp = np.linspace(
    max(tau_prior_min, best['tau_prior'] - tp_step),
    min(tau_prior_max, best['tau_prior'] + tp_step),
    10,
)
fine_eps = np.linspace(
    max(eps_min, best['epsilon'] - eps_step_size),
    min(eps_max, best['epsilon'] + eps_step_size),
    10,
)

total_fine = len(fine_tp) * len(fine_eps)
print(f'Refined grid: {len(fine_tp)} x {len(fine_eps)} = {total_fine} points')
print(f'  tau_prior: [{fine_tp[0]:.4f}, {fine_tp[-1]:.4f}]')
print(f'  epsilon:   [{fine_eps[0]:.6f}, {fine_eps[-1]:.6f}]')
print()

refined_results = grid_search_belief(records, fine_tp, fine_eps)
sweep_results.extend(refined_results)
best = pick_best(sweep_results, metric)
print(f'\nRefined best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Refined grid: 10 x 10 = 100 points
  tau_prior: [0.3211, 4.3011]
  epsilon:   [0.500000, 0.600000]

  [50/100] ...
  [100/100] ...

Refined best: tau_prior=2.5322 eps=0.544444
  combo_r=0.4847  marg_r=0.4030  mean_ll=-2.6210


In [7]:
# Scipy L-BFGS-B polishing
def objective(params):
    tp, eps = params
    res = evaluate_belief(records, tp, eps)
    return -res[metric]

tp_lo = max(tau_prior_min, best['tau_prior'] - tp_step / 2)
tp_hi = min(tau_prior_max, best['tau_prior'] + tp_step / 2)
eps_lo = max(eps_min, best['epsilon'] - eps_step_size / 2)
eps_hi = min(eps_max, best['epsilon'] + eps_step_size / 2)

x0 = [best['tau_prior'], best['epsilon']]
bounds = [(tp_lo, tp_hi), (eps_lo, eps_hi)]

print(f'Scipy optimization (metric={metric}) ...')
print(f'  bounds: tau_prior=[{tp_lo:.4f}, {tp_hi:.4f}]')
print(f'          epsilon=[{eps_lo:.6f}, {eps_hi:.6f}]')

opt_result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds,
                      options={'maxiter': 50, 'ftol': 1e-6})
opt_tp, opt_eps = opt_result.x
print(f'  Optimal: tau_prior={opt_tp:.4f} eps={opt_eps:.6f}')
print(f'  objective={-opt_result.fun:.4f}')

opt_eval = evaluate_belief(records, opt_tp, opt_eps)
sweep_results.append({**opt_eval, 'tau_prior': opt_tp, 'epsilon': opt_eps})
best = pick_best(sweep_results, metric)
print(f'\nFinal best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Scipy optimization (metric=combo_r) ...
  bounds: tau_prior=[1.5372, 3.5272]
          epsilon=[0.519444, 0.569444]
  Optimal: tau_prior=2.5330 eps=0.539879
  objective=0.4848

Final best: tau_prior=2.5330 eps=0.539879
  combo_r=0.4848  marg_r=0.4044  mean_ll=-2.6200


## Switch-Stage Fine-Tuning

In [8]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered


def evaluate_belief_switch(records, tau_prior, epsilon):
    """Run on full history, compute metrics only on switch stages."""
    full_results = run_predictions(records, make_bayesian_belief(
        tau_prior=tau_prior, epsilon=epsilon))
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_prior': tau_prior, 'epsilon': epsilon,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'epsilon': epsilon,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_switch_from_cache(records, trajectories):
    """Evaluate switch-stage metrics using precomputed trajectories."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(trajectories[idx])
    full_results = run_predictions(records, predict_fn)
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_belief_switch(records, tau_prior_vals, eps_vals):
    """Grid search optimizing switch-stage correlation."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    total = len(tau_prior_vals) * len(eps_vals)
    count = 0

    for tp in tau_prior_vals:
        for eps in eps_vals:
            count += 1
            if count % 50 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            trajectories = precompute_trajectories(records, tp, eps)
            res = evaluate_switch_from_cache(records, trajectories)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

In [9]:
# Coarse grid (same ranges)
print('=== Switch-Stage Fine-Tuning: Coarse Grid ===')
sw_total = len(tau_prior_vals) * len(eps_vals)
print(f'{sw_total} points\n')

sw_sweep = grid_search_belief_switch(records, tau_prior_vals, eps_vals)
best_sw = pick_best(sw_sweep, metric)
print(f'\nCoarse best: tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

=== Switch-Stage Fine-Tuning: Coarse Grid ===
210 points

  [50/210] ...
  [100/210] ...
  [150/210] ...
  [200/210] ...
  [210/210] ...

Coarse best: tau_prior=20.0000 eps=0.500000
  combo_r=0.4306  marg_r=0.5667  mean_ll=-3.0287


In [10]:
# Refined grid around switch-stage coarse best
fine_sw_tp = np.linspace(
    max(tau_prior_min, best_sw['tau_prior'] - tp_step),
    min(tau_prior_max, best_sw['tau_prior'] + tp_step),
    10,
)
fine_sw_eps = np.linspace(
    max(eps_min, best_sw['epsilon'] - eps_step_size),
    min(eps_max, best_sw['epsilon'] + eps_step_size),
    10,
)

total_sw_fine = len(fine_sw_tp) * len(fine_sw_eps)
print(f'Refined grid (switch-stage): {len(fine_sw_tp)} x {len(fine_sw_eps)} = {total_sw_fine} points')
print(f'  tau_prior: [{fine_sw_tp[0]:.4f}, {fine_sw_tp[-1]:.4f}]')
print(f'  epsilon:   [{fine_sw_eps[0]:.6f}, {fine_sw_eps[-1]:.6f}]')
print()

refined_sw = grid_search_belief_switch(records, fine_sw_tp, fine_sw_eps)
sw_sweep.extend(refined_sw)
best_sw = pick_best(sw_sweep, metric)
print(f'\nRefined best (switch): tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Refined grid (switch-stage): 10 x 10 = 100 points
  tau_prior: [18.0100, 20.0000]
  epsilon:   [0.450000, 0.550000]

  [50/100] ...
  [100/100] ...

Refined best (switch): tau_prior=20.0000 eps=0.483333
  combo_r=0.4308  marg_r=0.5614  mean_ll=-3.0834


In [11]:
# Scipy polishing for switch-stage tuning
def objective_sw(params):
    tp, eps = params
    res = evaluate_belief_switch(records, tp, eps)
    return -res[metric]

sw_tp_lo = max(tau_prior_min, best_sw['tau_prior'] - tp_step / 2)
sw_tp_hi = min(tau_prior_max, best_sw['tau_prior'] + tp_step / 2)
sw_eps_lo = max(eps_min, best_sw['epsilon'] - eps_step_size / 2)
sw_eps_hi = min(eps_max, best_sw['epsilon'] + eps_step_size / 2)

x0_sw = [best_sw['tau_prior'], best_sw['epsilon']]
bounds_sw = [(sw_tp_lo, sw_tp_hi), (sw_eps_lo, sw_eps_hi)]

print(f'Scipy optimization (switch-stage, metric={metric}) ...')
print(f'  bounds: tau_prior=[{sw_tp_lo:.4f}, {sw_tp_hi:.4f}]')
print(f'          epsilon=[{sw_eps_lo:.6f}, {sw_eps_hi:.6f}]')

opt_sw = minimize(objective_sw, x0_sw, method='L-BFGS-B', bounds=bounds_sw,
                  options={'maxiter': 50, 'ftol': 1e-6})
sw_tp, sw_eps = opt_sw.x
print(f'  Optimal: tau_prior={sw_tp:.4f} eps={sw_eps:.6f}')
print(f'  objective={-opt_sw.fun:.4f}')

opt_sw_eval = evaluate_belief_switch(records, sw_tp, sw_eps)
sw_sweep.append({**opt_sw_eval, 'tau_prior': sw_tp, 'epsilon': sw_eps})
best_sw = pick_best(sw_sweep, metric)
print(f'\nFinal best (switch): tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Scipy optimization (switch-stage, metric=combo_r) ...
  bounds: tau_prior=[19.0050, 20.0000]
          epsilon=[0.458333, 0.508333]
  Optimal: tau_prior=20.0000 eps=0.486602
  objective=0.4308

Final best (switch): tau_prior=20.0000 eps=0.486602
  combo_r=0.4308  marg_r=0.5624  mean_ll=-3.0720


## Save Parameters

In [12]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'tau_prior': best['tau_prior'], 'epsilon': best['epsilon'],
        'combo_r': best['combo_r'], 'marg_r': best['marg_r'], 'mean_ll': best['mean_ll'],
    },
    'switch_stage_tuned': {
        'tau_prior': best_sw['tau_prior'], 'epsilon': best_sw['epsilon'],
        'combo_r': best_sw['combo_r'], 'marg_r': best_sw['marg_r'], 'mean_ll': best_sw['mean_ll'],
    },
}

with open('bayesian_belief_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to bayesian_belief_params.json')
print(json.dumps(output, indent=2))

Saved to bayesian_belief_params.json
{
  "metric_optimized": "combo_r",
  "aggregate_tuned": {
    "tau_prior": 2.5329620514938,
    "epsilon": 0.5398790489920662,
    "combo_r": 0.484759578734252,
    "marg_r": 0.40438947417668447,
    "mean_ll": -2.6200474279292982
  },
  "switch_stage_tuned": {
    "tau_prior": 20.0,
    "epsilon": 0.4866017196491444,
    "combo_r": 0.4308031448634344,
    "marg_r": 0.5624062698272309,
    "mean_ll": -3.0720064552721227
  }
}
